In [ ]:
from argparse import Namespace
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.build import build_dataset
from src.misc import load_config
from src.predict import load_predict_config, load_predictor


In [ ]:
PREDICT_CONFIG = ROOT / "config" / "predict.yaml"
SAMPLE_COUNT = 4
AXIS = 0
AXES = ("axis 0", "axis 1", "axis 2")

In [ ]:
cfg = load_predict_config(PREDICT_CONFIG)
run = ROOT / cfg.run_dir
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pred = load_predictor(run, device=dev)
opts = cfg.make_options(pred)

data = Namespace(**load_config(run / "model.yaml"))
data.data_dir = {
    axis: (path if (path := Path(value)).is_absolute() else run / path).resolve()
    for axis, value in data.data_dir.items()
}
data.augment = False
ds = build_dataset(data)
pool = ds.condition_indices[AXIS]
pick = torch.randint(len(pool), (SAMPLE_COUNT,)).tolist()
real = torch.stack([ds[pool[i]][0][0] for i in pick])

vol, stats = pred.predict(opts)
idx = torch.linspace(0, opts.volume_size - 1, SAMPLE_COUNT).round().int().tolist()
gen = torch.stack([vol.select(AXIS, i) for i in idx])
fracs = [round(v, 4) for v in stats["phase_fractions"]]
print(f"run={run.name} sampler={stats['sampler']} fractions={fracs}")

In [ ]:
fig, axes = plt.subplots(
    2, SAMPLE_COUNT, figsize=(3 * SAMPLE_COUNT, 8), squeeze=False
)
for row, (imgs, title) in enumerate(
    (
        (real, f"TRAINING DATA / {AXES[AXIS]}"),
        (gen, f"GENERATED / {AXES[AXIS]}"),
    )
):
    for col, (ax, img) in enumerate(zip(axes[row], imgs)):
        ax.imshow(
            img,
            cmap="gray",
            vmin=0,
            vmax=pred.num_phases - 1,
            interpolation="nearest",
        )
        ax.set_title(
            f"{title} {col + 1}", fontsize=16, fontweight="bold", pad=10
        )
        ax.axis("off")
fig.suptitle(
    "Training data vs generated samples", fontsize=22, fontweight="bold", y=0.97
)
fig.subplots_adjust(left=0.03, right=0.99, bottom=0.03, top=0.88, wspace=0.10, hspace=0.35)
plt.show()